# 推理 Infra 学习实录 05：从 nano-vLLM 到 mini-SGLang

> 归档状态（GitHub 发布副本）：流程草稿，尚未实跑，不提供性能结论。 所有 ModelScope 发布 URL 尚未知，不表示已发布。
> [配套文章](article.md) · [上一篇](../04/article.md) · [下一篇](../06/article.md)。Notebook 输出与执行计数已清空。
> 版本边界：nano-vLLM 安装与源码说明对齐已核对 commit `bb823b3e06983d71485a8e1f23715ebd87d98ef8`；已有 checkout 不自动覆盖，版本不符时应另选空目录。
> 路径说明：`/mnt/workspace` 是 ModelScope 平台缓存示例，请按实际环境调整，模型目录以 `snapshot_download()` 返回值为准。

这份 Notebook 计划完成 mini-SGLang 的基础运行，并将其 Radix Cache、Chunked Prefill、Overlap Scheduling 和服务结构与 nano-vLLM 对照。

本篇不做性能排名。没有相同硬件、相同依赖、相同 workload 和多轮预热的数据，不足以支持速度结论。

## 1. 环境检查

mini-SGLang 依赖 Linux、NVIDIA CUDA、FlashInfer 和 `sgl-kernel`。先确认当前 Notebook 环境。

In [ ]:
!nvidia-smi

In [ ]:
import platform
import sys
import torch

print("Platform:", platform.platform())
print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Torch CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

## 2. 使用独立虚拟环境

mini-SGLang 当前约束 `torch<2.10.0` 和特定 Transformers 版本。独立 `.venv-minisgl` 用于隔离依赖，避免修改 Notebook kernel 中已有的 nano-vLLM 环境。

In [ ]:
%pip install -q uv modelscope pandas

本文核对的官方源码 commit 为 `9a91cfafe754aa85daee49998176275667eb58f2`。固定 commit 可以避免 Notebook 内容随 main 分支变化。

首次克隆使用当前工作目录下的 `mini-sglang/`，环境统一为 `mini-sglang/.venv-minisgl`。已有 checkout 只检查、不覆盖；本文归档修订不会执行 clone/install。

In [ ]:
from pathlib import Path
import subprocess

repo_dir = Path("mini-sglang")
commit = "9a91cfafe754aa85daee49998176275667eb58f2"

if not repo_dir.exists():
    subprocess.run(
        ["git", "clone", "https://github.com/sgl-project/mini-sglang.git", str(repo_dir)],
        check=True,
    )
    subprocess.run(["git", "-C", str(repo_dir), "checkout", commit], check=True)
else:
    print("保留已有 checkout，不自动 fetch、pull 或覆盖：", repo_dir)
actual_commit = subprocess.check_output(
    ["git", "-C", str(repo_dir), "rev-parse", "HEAD"], text=True,
).strip()
if actual_commit != commit:
    raise RuntimeError(f"源码版本不符：{actual_commit}；请另选空目录克隆固定版本。")
print("source:", repo_dir.absolute())
print("commit:", actual_commit)

In [ ]:
venv_dir = repo_dir / ".venv-minisgl"
venv_python = venv_dir / "bin" / "python"

if not venv_python.exists():
    subprocess.run(["uv", "venv", "--python", "3.12", str(venv_dir)], check=True)

subprocess.run(
    ["uv", "pip", "install", "--python", str(venv_python), "-e", str(repo_dir)],
    check=True,
)
print("isolated python:", venv_python.resolve())

## 3. 验证安装与 CLI

版本检查在隔离环境中执行。CLI 输出同时用于确认当前参数名称。

In [ ]:
check_code = """
import torch
import transformers
import minisgl
print('torch:', torch.__version__)
print('transformers:', transformers.__version__)
print('minisgl:', minisgl.__file__)
print('cuda available:', torch.cuda.is_available())
"""
subprocess.run([str(venv_python), "-c", check_code], check=True)

In [ ]:
help_result = subprocess.run(
    [str(venv_python), "-m", "minisgl", "--help"],
    check=True,
    text=True,
    capture_output=True,
)
for line in help_result.stdout.splitlines():
    if any(key in line for key in ["model-source", "cache-type", "max-prefill", "page-size"]):
        print(line)

当前源码使用 `--cache-type radix|naive`。如果 README 示例与 `--help` 不一致，以固定 commit 的 CLI 和 `server/args.py` 为准。

## 4. 通过 ModelScope 下载 Qwen3-0.6B

离线 `LLM` 接口接收本地模型目录。显式下载后，模型路径也可以复用于其他推理框架。

In [ ]:
from modelscope import snapshot_download

model_dir = snapshot_download(
    "Qwen/Qwen3-0.6B",
    cache_dir="/mnt/workspace/.cache/modelscope",
)
print("model_dir:", model_dir)

## 5. 最小离线生成

模型在独立进程中初始化。进程结束后 GPU 资源会完整释放，后续实验不会与当前 Notebook kernel 中的其他引擎争抢显存。

In [ ]:
offline_code = r'''
import sys
from minisgl.core import SamplingParams
from minisgl.llm import LLM

model_dir = sys.argv[1]
llm = LLM(
    model_dir,
    max_seq_len_override=4096,
    max_extend_tokens=2048,
    cuda_graph_max_bs=8,
    page_size=16,
)
prompt = llm.tokenizer.apply_chat_template(
    [{"role": "user", "content": "请用三句话解释什么是大模型推理引擎。"}],
    tokenize=False,
    add_generation_prompt=True,
)
outputs = llm.generate(
    [prompt],
    SamplingParams(temperature=0.6, max_tokens=128),
)
print(outputs[0]["text"])
print("output tokens:", len(outputs[0]["token_ids"]))
llm.shutdown()
'''
subprocess.run([str(venv_python), "-c", offline_code, model_dir], check=True)

## 6. 对比离线 API

两个框架都提供 `LLM`、`SamplingParams` 和 `generate()`，但入口背后的系统边界不同。

In [ ]:
import pandas as pd

api_comparison = pd.DataFrame([
    ("入口", "nanovllm.LLM", "minisgl.llm.LLM"),
    ("请求状态", "Sequence", "PendingReq / Req / Batch"),
    ("Prefill 管理", "Scheduler.waiting", "PrefillManager"),
    ("Decode 管理", "Scheduler.running", "DecodeManager"),
    ("KV 管理", "BlockManager", "CacheManager + PrefixCache"),
    ("模型执行", "ModelRunner", "Engine"),
    ("在线服务", "未内置", "OpenAI-compatible API"),
], columns=["维度", "nano-vLLM", "mini-SGLang"])
api_comparison

## 7. 查看 Radix Cache 源码入口

核心方法包括 `match_prefix()`、`insert_prefix()`、`lock_handle()`、`evict()` 和 `_tree_walk()`。

In [ ]:
radix_path = repo_dir / "python/minisgl/kvcache/radix_cache.py"
radix_source = radix_path.read_text()

for symbol in ["class RadixTreeNode", "class RadixPrefixCache", "def match_prefix", "def insert_prefix", "def evict", "def _tree_walk"]:
    print(symbol, "->", radix_source.find(symbol))

Radix Tree 显式表达共享前缀和分支；nano-vLLM 使用链式 hash 隐式表达完整前缀。两种实现都要求从请求开头连续匹配。

In [ ]:
cache_comparison = pd.DataFrame([
    ("索引结构", "链式 hash", "Radix Tree"),
    ("映射结构", "Sequence.block_table", "TableManager.page_table"),
    ("匹配边界", "完整 block", "page 对齐前缀"),
    ("共享保护", "Block.ref_count", "node.ref_count + lock/unlock"),
    ("空间回收", "free/used block", "evictable/protected + leaf eviction"),
    ("部分节点匹配", "不拆分 block", "split_at()"),
], columns=["维度", "nano-vLLM", "mini-SGLang"])
cache_comparison

## 8. 定位 Chunked Prefill

mini-SGLang 使用 `ChunkedReq` 表示未完成 prefill 的请求，并通过 `max_extend_tokens` 控制本轮 token budget。

In [ ]:
prefill_path = repo_dir / "python/minisgl/scheduler/prefill.py"
prefill_source = prefill_path.read_text()

for symbol in ["class ChunkedReq", "class PrefillAdder", "chunk_size =", "class PrefillManager"]:
    print(symbol, "->", prefill_source.find(symbol))

In [ ]:
prefill_comparison = pd.DataFrame([
    ("状态对象", "Sequence 留在 waiting", "ChunkedReq"),
    ("token budget", "max_num_batched_tokens", "max_extend_tokens"),
    ("前缀命中进度", "num_cached_tokens", "cached_len"),
    ("未来输出预留", "简化处理", "extend_len + output_len"),
    ("在途 decode 影响", "未显式纳入该预算", "inflight_tokens 计入 reserved_size"),
], columns=["维度", "nano-vLLM", "mini-SGLang"])
prefill_comparison

## 9. 定位 Overlap Scheduling

`overlap_loop()` 在 GPU 执行当前 batch 时处理上一批结果，用于隐藏部分 CPU 调度和元数据处理开销。

In [ ]:
scheduler_path = repo_dir / "python/minisgl/scheduler/scheduler.py"
scheduler_source = scheduler_path.read_text()

for symbol in ["def overlap_loop", "def normal_loop", "def run_forever", "def _schedule_next_batch"]:
    print(symbol, "->", scheduler_source.find(symbol))

Overlap Scheduling 的性能影响需要单独做消融实验。环境变量 `MINISGL_DISABLE_OVERLAP_SCHEDULING=1` 可关闭该路径，但本篇不使用单次运行结果建立性能结论。

## 10. OpenAI-compatible API 启动方式

以下命令用于独立终端。服务会持续占用当前进程，不应作为普通 Notebook 单元直接阻塞执行。

```bash
mini-sglang/.venv-minisgl/bin/python -m minisgl \
  --model Qwen/Qwen3-0.6B \
  --model-source modelscope \
  --host 127.0.0.1 \
  --port 1919
```

当前 CLI 使用 `--cache-type radix` 或 `--cache-type naive`。

回环地址仅限制可达范围，不是鉴权；对外提供服务前需配置访问控制。

In [ ]:
server_command = [
    str(venv_python), "-m", "minisgl",
    "--model", "Qwen/Qwen3-0.6B",
    "--model-source", "modelscope",
    "--host", "127.0.0.1",
    "--port", "1919",
]
print(" ".join(server_command))

## 11. 学习路径总结

```text
nano-vLLM
-> generate / scheduler / block manager / attention
-> 理解单卡推理引擎核心链路

mini-SGLang
-> Radix Cache / Chunked Prefill / Overlap Scheduling / API Server
-> 理解在线 serving 系统结构
```

下一篇将基于 OpenAI-compatible API 构建魔搭创空间 Demo。